# Structured Output Deep Dive

Get validated Pydantic objects from Workers AI — even with deeply nested schemas that break tool calling.

This notebook covers:
1. Why tool calling fails on complex schemas
2. How `cf_structured()` solves it
3. Simple → medium → complex schema examples with real outputs
4. AI Gateway caching + prompt caching
5. Multi-model comparison

In [1]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

## 1. Simple schema — works with `cloudflare_agent()`

In [2]:
from pydantic import BaseModel
from pydantic_ai_cloudflare import cloudflare_agent

class Language(BaseModel):
    name: str
    type: str
    year: int
    features: list[str]

agent = cloudflare_agent(output_type=Language)
r = agent.run_sync("Tell me about Python")
print(f"Name: {r.output.name}")
print(f"Type: {r.output.type}")
print(f"Year: {r.output.year}")
print(f"Features: {r.output.features}")

Name: Python
Type: Programming Language
Year: 1991
Features: ['Dynamic typing', 'Garbage collection', 'Extensive library']


## 2. Complex schema — use `cf_structured()`

Tool calling breaks on schemas with 3+ nested models, `Literal` types, and `Optional` fields. `cf_structured()` uses schema injection + `json_object` mode instead.

In [3]:
from typing import Literal, Optional
from pydantic import Field
from pydantic_ai_cloudflare import cf_structured_sync

class NewsItem(BaseModel):
    date: Optional[str] = None
    headline: str
    summary: str = ""
    source: str = "Web"

class TriggerEvent(BaseModel):
    event: str
    priority: Literal["HIGH", "MEDIUM", "LOW"] = "MEDIUM"
    source: str = "Web"

class HiringSignal(BaseModel):
    role: str
    strategic_insight: str = ""

class MarketIntelligence(BaseModel):
    recent_news: list[NewsItem] = Field(default_factory=list)
    trigger_events: list[TriggerEvent] = Field(default_factory=list)
    hiring_signals: list[HiringSignal] = Field(default_factory=list)

class Report(BaseModel):
    market_intelligence: MarketIntelligence

print("Schema: 3 nested models, Literal['HIGH','MEDIUM','LOW']")

result = cf_structured_sync(
    "Format research for a fictional logistics company 'SwiftFreight LLC'.\n"
    "News: [2025-11] Last-mile expansion. [2025-10] New CCO. "
    "[2025-05] AI network launched. [2025-05] Procurement platform. "
    "[2025-05] EV truck program. [2026-04] Cross-border constraints.\n"
    "Triggers: New CCO (HIGH), New CEO (HIGH), 2022 security incident (MEDIUM), "
    "SOC certs (HIGH), Multi-cloud migration (HIGH), Hybrid work (MEDIUM).\n"
    "Generate 6 news, 6 triggers, 2 hiring signals.",
    Report,
    max_tokens=8192,
)

mi = result.market_intelligence
print(f"\nNews ({len(mi.recent_news)}):")
for n in mi.recent_news:
    print(f"  [{n.date}] {n.headline[:60]}")
print(f"\nTriggers ({len(mi.trigger_events)}):")
for t in mi.trigger_events:
    print(f"  [{t.priority}] {t.event[:60]}")
print(f"\nHiring ({len(mi.hiring_signals)}):")
for h in mi.hiring_signals:
    print(f"  {h.role}: {h.strategic_insight[:50]}")

Schema: 3 nested models, Literal['HIGH','MEDIUM','LOW']

News (6):
  [2025-11] Expansion of last-mile delivery capabilities
  [2025-10] New Chief Commercial Officer appointed
  [2025-05] AI logistics network launched at scale
  [2025-05] Procurement platform Exchange: Spot launched
  [2025-05] EV truck program launched with Tesla partnership
  [2026-04] Cross-border capacity constraints reported

Triggers (6):
  [HIGH] New CCO appointed for commercial growth
  [HIGH] New CEO appointed, founder transitions to Chairman
  [MEDIUM] Parent company had major security incident in 2022
  [HIGH] Active SOC 1/2 certifications maintained
  [HIGH] Multi-cloud migration to GCP/OCI/AWS
  [MEDIUM] Hybrid work policy implemented

Hiring (2):
  Cloud Security Engineer: Improving security posture
  Data Scientist: Enhancing AI capabilities


## 3. Why tool calling fails

LangChain and PydanticAI send the schema as a `tools` array:
```json
{"tools": [{"type": "function", "function": {"name": "MySchema", "parameters": {huge nested schema}}}]}
```

Workers AI returns `function.arguments = null` for complex schemas. The retry mechanism sends array-content messages that Workers AI rejects with 400.

`cf_structured()` avoids this entirely:
- Schema goes in the **system prompt** (not tools array)
- `response_format: json_object` forces valid JSON
- Retries send error feedback as a **user message** (not the broken format)

## 4. AI Gateway — logging + caching

Route through AI Gateway for free logging, analytics, and response caching.

In [ ]:
# With AI Gateway (logging + 5-minute cache)
result = cf_structured_sync(
    "Describe Python in 3 bullet points",
    Language,
    gateway_id="default",    # auto-creates on first call
    cache_ttl=300,            # cache for 5 minutes
)

# Second call with same prompt → cached response (~instant)
result2 = cf_structured_sync(
    "Describe Python in 3 bullet points",
    Language,
    gateway_id="default",
    cache_ttl=300,
)
# Check dash.cloudflare.com → AI → AI Gateway for logs

## 5. Prompt caching (session affinity)

For multi-turn conversations, `session_id` routes to the same model instance so the KV prefix cache hits.

In [ ]:
# First call: full inference
r1 = cf_structured_sync(
    "Tell me about Rust", Language,
    session_id="my-session-001",
)

# Follow-up: same session → KV cache hit, faster
r2 = cf_structured_sync(
    "Now tell me about Go", Language,
    session_id="my-session-001",
)

## 6. Schema complexity check

Before running, check if your schema will work reliably.

In [4]:
from pydantic_ai_cloudflare import schema_stats

stats = schema_stats(Report)
print(f"Schema: {stats['total_chars']} chars, {stats['nested_model_count']} nested models")
print(f"Simplified: {stats['simplified_chars']} chars ({stats['reduction']} reduction)")
print(f"Recommendation: {stats['recommendation']}")

Schema: 1567 chars, 4 nested models
Simplified: 1201 chars (23% reduction)
Recommendation: Good -- should work reliably with any Workers AI model.


## Summary

| Scenario | Method | Works? |
|---|---|---|
| Simple (3-5 fields) | `cloudflare_agent(output_type=X)` | Yes |
| Complex (4+ nested, Literal) | `cf_structured()` | Yes — all 6 models |
| With logging | `cf_structured(gateway_id='default')` | Yes |
| With caching | `cf_structured(cache_ttl=300)` | Yes |
| With prompt caching | `cf_structured(session_id='...')` | Yes |
| LangChain tool calling | `with_structured_output()` | Fails on complex schemas |